In [1]:

#importing necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# First Model:
## Step 1: Loading the Data
We import our cleaned_data with all data needed

In [2]:
df  = pd.read_csv('cleaned_dataset.csv', sep=';')

## Step 2: Preparing Time Data
We transform our date value present on our dataset in a usable format because a YEAR-MONTH format isn't readable by our future model. We need to devide it into 2 simple values (year & month)

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
# We create a new 'Month' column by extracting only the month
df['Month'] = df['Date'].dt.month
# We create a new 'Year' column by extracting only the year
df['Year'] = df['Date'].dt.year

## Step 3: Defining the Goal (Target) and what is usable (Features)
To train an AI, we must tell it what it needs to guess (the **Target**) and what information it is allowed to use (the **Features**). 

Then, we clean the table to remove rows where the delay is missing.

In [4]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Number of scheduled trains', 'Month', 'Year']

# We remove rows where the delay is equal to -1 (meaning the data was invalid during cleaning).
df = df[df[target] != -1]


# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 4: Translating Text for the Computer (Encoding)
An Artificial Intelligence is a calculating machine: it only understands mathematics. It cannot read words like "Paris" or "Bordeaux". 
We must therefore use a "translator" to convert these station names into numerical codes that the computer can analyze. Encoding of columns where the value isn't usable by the model (strings) into and usable value

In [5]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 5: Creating the AI "Assembly Line"
We are going to set up our prediction model. The chosen algorithm is called a **Random Forest**. 
It combines the output of multiple decision trees to reach a single result

In [6]:
# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 6: Creating the final AI
This is the most important step. We will divide our data into two batches:
- **80% for training:** The AI practices guessing the delays and looks at the real answers to learn from its mistakes.
- **20% for testing:** We hide the answers from the AI and ask it to make its predictions to see if it has understood the logic.

In [7]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 7: Grading (Performance Evaluation)
Now that the AI has taken its exam and made its predictions, we will compare its answers with reality to give it performance grades.

In [8]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")

Mean Absolute Error (MAE) : 2.04 minutes
Mean Squared Error (MSE) : 14.43
R² Score : 0.13


## Conclusion on the first model
For a first model, it's not bad but it's not good either. A R² score of 0.13 mean that the model is slightly better than pur randomness.
This mean that this model can be tuned even more to try to have a better score and so a better prediction model.